# 00 - Data Wrangling
## Poland Apartments (Multi-City) — Sale Listings

**Data source:** Kaggle — `krzysztofjamroz/apartment-prices-in-poland`
**Scope:** 15 Polish cities, monthly snapshots Aug 2023 – Jun 2024
**Design decision:** Keep ALL monthly snapshots per listing (not just the latest) to preserve temporal price trends. This means the same physical apartment (`id`) can appear multiple times — this is intentional, not a duplicate to be removed. See the "Cross-month overlap" section below for why.

**Important:** because the same `id` can appear in multiple rows, any later train/test split MUST be grouped by `id` (e.g. `GroupShuffleSplit`), not a plain random split — otherwise the same physical apartment could leak between train and test.

In [1]:
import pandas as pd
import numpy as np
import glob
import re
import os

pd.set_option('display.max_columns', None)

In [5]:
data_dir = 'C:/Users/piyub/real_estate_price_prediction/flat_data' 
import os
print(os.getcwd())
print(os.listdir(data_dir))

C:\Users\piyub\real_estate_price_prediction
['apartments_pl_2023_08.csv', 'apartments_pl_2023_09.csv', 'apartments_pl_2023_10.csv', 'apartments_pl_2023_11.csv', 'apartments_pl_2023_12.csv', 'apartments_pl_2024_01.csv', 'apartments_pl_2024_02.csv', 'apartments_pl_2024_03.csv', 'apartments_pl_2024_04.csv', 'apartments_pl_2024_05.csv', 'apartments_pl_2024_06.csv']


## 1. Load & combine all monthly sale files, tagging each row with its snapshot month

In [6]:
data_dir = 'C:/Users/piyub/real_estate_price_prediction/flat_data' 

files = sorted(glob.glob(os.path.join(data_dir, 'apartments_pl_*.csv')))
print(f"Found {len(files)} sale files")

dfs = []
for f in files:
    df = pd.read_csv(f)
    match = re.search(r'(\d{4})_(\d{2})', os.path.basename(f))
    year, month = match.groups()
    df['snapshot_month'] = f'{year}-{month}'
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(raw.shape)
raw.head()

Found 11 sale files
(195568, 29)


,id,city,type,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,centreDistance,poiCount,schoolDistance,clinicDistance,postOfficeDistance,kindergartenDistance,restaurantDistance,collegeDistance,pharmacyDistance,ownership,buildingMaterial,condition,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom,price,snapshot_month
0,f8524536d4b09a0c8ccc0197ec9d7bde,szczecin,blockOfFlats,63.00,3.0,4.0,10.0,1980.0,53.378933,14.625296,6.53,9.0,0.118,1.389,0.628,0.105,1.652,NaN,0.413,condominium,concreteSlab,NaN,yes,yes,yes,no,yes,415000,2023-08
1,accbe77d4b360fea9735f138a50608dd,szczecin,blockOfFlats,36.00,2.0,8.0,10.0,NaN,53.442692,14.559690,2.15,16.0,0.273,0.492,0.652,0.291,0.348,1.404,0.205,cooperative,concreteSlab,NaN,no,yes,yes,no,yes,395995,2023-08
2,8373aa373dbc3fe7ca3b7434166b8766,szczecin,tenement,73.02,3.0,2.0,3.0,NaN,53.452222,14.553333,3.24,9.0,0.275,0.672,0.367,0.246,0.300,1.857,0.280,condominium,brick,NaN,no,no,no,no,no,565000,2023-08
3,0a68cd14c44ec5140143ece75d739535,szczecin,tenement,87.60,3.0,2.0,3.0,NaN,53.435100,14.532900,2.27,32.0,0.175,0.259,0.223,0.359,0.101,0.310,0.087,condominium,brick,NaN,yes,yes,no,no,yes,640000,2023-08
4,f66320e153c2441edc0fe293b54c8aeb,szczecin,blockOfFlats,66.00,3.0,1.0,3.0,NaN,53.410278,14.503611,4.07,1.0,0.218,1.690,0.504,0.704,0.501,2.138,0.514,condominium,NaN,NaN,no,no,no,no,no,759000,2023-08


## 2. Cross-month overlap check

Confirms the same `id` legitimately represents the same physical unit across months (stable `squareMeters`/`rooms`), and that price does move over time for a meaningful share of listings — this is the temporal signal the old single-snapshot project couldn't capture.

In [7]:
print("Total rows:", len(raw))
print("Unique listings (ids):", raw['id'].nunique())

multi = raw.groupby('id').filter(lambda g: len(g) > 1)
print("Listings appearing in >1 month:", multi['id'].nunique())

stability = multi.groupby('id')[['squareMeters', 'rooms']].nunique()
print("ids with inconsistent squareMeters (data noise, investigate):", (stability['squareMeters'] > 1).sum())

price_range = multi.groupby('id')['price'].nunique()
print("ids where price changed at least once:", (price_range > 1).sum())

Total rows: 195568
Unique listings (ids): 92967
Listings appearing in >1 month: 45347
ids with inconsistent squareMeters (data noise, investigate): 1181
ids where price changed at least once: 12937


Listings appearing in >1 month: 45347
ids with inconsistent squareMeters (data noise, investigate): 1181
ids where price changed at least once: 12937


In [8]:
# Drop the small number of ids with inconsistent squareMeters — likely different
# physical units that were assigned the same id by mistake in the source data,
# not something we want to treat as one property's price history.
bad_ids = stability[stability['squareMeters'] > 1].index
raw = raw[~raw['id'].isin(bad_ids)].copy()
print("Rows after dropping inconsistent ids:", len(raw))

Rows after dropping inconsistent ids: 190858


## 3. Basic filtering

In [9]:
# No outlier filtering here — deferred to the modeling notebook, computed on the
# TRAIN split only, to avoid leaking test-set distribution into cleaning thresholds
# (this is a fix vs. the original Cracow-only project, which filtered before splitting).

print("Rows before filtering:", len(raw))
raw = raw.dropna(subset=['price', 'squareMeters', 'rooms', 'city'])
print("Rows after dropping rows missing core fields:", len(raw))

Rows before filtering: 190858
Rows after dropping rows missing core fields: 190858


## 4. Missing value handling for categoricals

`condition` (76% missing), `buildingMaterial` (39% missing), and `type` (21% missing) are too sparse to impute reliably — filled with an explicit `"unknown"` category instead, so the model can still learn from "this info wasn't disclosed" as a signal, without fabricating values.

In [10]:
for col in ['condition', 'buildingMaterial', 'type']:
    raw[col] = raw[col].fillna('unknown')

raw[['condition', 'buildingMaterial', 'type']].apply(pd.Series.value_counts)

,condition,buildingMaterial,type
apartmentBuilding,NaN,NaN,31565.0
blockOfFlats,NaN,NaN,89443.0
brick,NaN,89444.0,NaN
concreteSlab,NaN,25845.0,NaN
low,21012.0,NaN,NaN
premium,27219.0,NaN,NaN
tenement,NaN,NaN,28731.0
unknown,142627.0,75569.0,41119.0


## 5. Boolean columns

`yes`/`no` string columns converted to actual booleans.

In [11]:
bool_cols = ['hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']
for col in bool_cols:
    raw[col] = raw[col].map({'yes': True, 'no': False})

# hasElevator retains NaN for genuinely unknown cases (4.9% missing) — left as-is here,
# handled via imputation in the modeling notebook (train-fit only).
raw[bool_cols].isna().mean()

hasParkingSpace    0.000000
hasBalcony         0.000000
hasElevator        0.050053
hasSecurity        0.000000
hasStorageRoom     0.000000
dtype: float64

## 6. Numeric fields with moderate missingness

`floor`, `floorCount`, `buildYear` — left as NaN here deliberately. KNN imputation happens in the modeling notebook, fit on the training split only, to avoid leaking test-set information into imputed values.

In [12]:
raw[['floor', 'floorCount', 'buildYear']].isna().mean()

floor         0.176671
floorCount    0.012166
buildYear     0.164766
dtype: float64

## 7. Drop uninformative / redundant columns

In [13]:
# Keep 'id' — needed downstream for GroupShuffleSplit so the same physical
# apartment can't appear in both train and test.
# Drop lat/long directly; centreDistance + poiCount already summarize location
# usefully, and raw coordinates aren't usable without a clustering/embedding step.
cleaned = raw.drop(columns=['latitude', 'longitude'])
cleaned.shape

(190858, 27)

## 8. Save cleaned dataset

In [14]:
os.makedirs(data_dir, exist_ok=True)
cleaned.to_csv(os.path.join(data_dir, 'cleaned_data_sale.csv'), index=False)
print("Saved:", cleaned.shape)
cleaned.head()

Saved: (190858, 27)


,id,city,type,squareMeters,rooms,floor,floorCount,buildYear,centreDistance,poiCount,schoolDistance,clinicDistance,postOfficeDistance,kindergartenDistance,restaurantDistance,collegeDistance,pharmacyDistance,ownership,buildingMaterial,condition,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom,price,snapshot_month
0,f8524536d4b09a0c8ccc0197ec9d7bde,szczecin,blockOfFlats,63.00,3.0,4.0,10.0,1980.0,6.53,9.0,0.118,1.389,0.628,0.105,1.652,NaN,0.413,condominium,concreteSlab,unknown,True,True,True,False,True,415000,2023-08
1,accbe77d4b360fea9735f138a50608dd,szczecin,blockOfFlats,36.00,2.0,8.0,10.0,NaN,2.15,16.0,0.273,0.492,0.652,0.291,0.348,1.404,0.205,cooperative,concreteSlab,unknown,False,True,True,False,True,395995,2023-08
2,8373aa373dbc3fe7ca3b7434166b8766,szczecin,tenement,73.02,3.0,2.0,3.0,NaN,3.24,9.0,0.275,0.672,0.367,0.246,0.300,1.857,0.280,condominium,brick,unknown,False,False,False,False,False,565000,2023-08
3,0a68cd14c44ec5140143ece75d739535,szczecin,tenement,87.60,3.0,2.0,3.0,NaN,2.27,32.0,0.175,0.259,0.223,0.359,0.101,0.310,0.087,condominium,brick,unknown,True,True,False,False,True,640000,2023-08
4,f66320e153c2441edc0fe293b54c8aeb,szczecin,blockOfFlats,66.00,3.0,1.0,3.0,NaN,4.07,1.0,0.218,1.690,0.504,0.704,0.501,2.138,0.514,condominium,unknown,unknown,False,False,False,False,False,759000,2023-08


## Notes / decisions made in this notebook

- **Kept all monthly snapshots** (not deduped to latest-per-id) to preserve temporal price signal — 12,937 listings show a genuine price change across snapshots (median 4.3% change).
- **No outlier filtering or imputation fit here** — both deferred to the modeling notebook and computed on the training split only, to avoid data leakage.
- **`id` retained** — required for grouped train/test splitting downstream (`GroupShuffleSplit` on `id`), since the same apartment can appear in multiple rows.
- **Sparse categoricals filled with `"unknown"`** rather than dropped or imputed, since `condition` alone is 76% missing — dropping would lose the majority of the dataset.